In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import subprocess
import os
# python version: 3.11.9

In [ ]:
# presets, functions

# preset values for the analysis:
#____________________________________________________________________________________________________________________
# physical constants
Z = 6
A = 12
mass_nucleon = 0.938273 
# mass_nucleus = 11.178
mass_nucleus = A * 0.931494
alpha_fine = 1 / 137

# three-momentum bin centers
qvcenters = [0.100, 0.148, 0.167, 0.205, 0.240, 0.300, 0.380, 0.475, 0.570, 0.649, 0.756, 0.991, 1.619, 1.921, 2.213, 2.500, 2.783, 3.500]
# three-momentum edges
qvbins = [0.063, 0.124, 0.158, 0.186, 0.223, 0.270, 0.340, 0.428, 0.523, 0.609, 0.702, 0.878, 1.302, 1.770, 2.067, 2.357, 2.642, 2.923, 4.500]
# four-momentum squared bin names, in string format
qvbin_names = ['[0.063,0.124]', '[0.124,0.158]', '[0.158,0.186]', '[0.186,0.223]', '[0.223,0.270]', '[0.270,0.340]', '[0.340,0.428]', '[0.428,0.523]', '[0.523,0.609]',
                '[0.609,0.702]', '[0.702,0.878]', '[0.878,1.302]', '[1.302,1.770]', '[1.770,2.067]', '[2.067,2.357]', '[2.357,2.642]', '[2.642,2.923]', '[2.923,4.500]']

# four-momentum squared bin centers
Q2centers = [0.010, 0.020, 0.026, 0.040, 0.056, 0.093, 0.120, 0.160, 0.265, 0.380, 0.500, 0.800, 1.250, 1.750, 2.250, 2.750, 3.250, 3.750]
# four-momentum squared bin edges
Q2bins = [0.004, 0.015, 0.025, 0.035, 0.045, 0.070, 0.100, 0.145, 0.206, 0.322, 0.438, 0.650, 1.050, 1.500, 2.000, 2.500, 3.000, 3.500, 4.000]
# four-momentum squared bin names, in string format
Q2bin_names = ['[0.004,0.015]', '[0.015,0.025]', '[0.025,0.035]', '[0.035,0.045]', '[0.045,0.070]', '[0.070,0.100]', '[0.100,0.145]', '[0.145,0.206]', '[0.206,0.322]',
                '[0.322,0.438]', '[0.438,0.650]', '[0.650,1.050]', '[1.050,1.500]', '[1.500,2.000]', '[2.000,2.500]', '[2.500,3.000]', '[3.000,3.500]', '[3.500,4.000]']

dataSet_to_normalization = {
    1: 0.95971, 2: 0.96416, 3: 1.0744, 4: 0.99482, 5: 0.93381,
    6: 1.0126, 7: 0.96716, 8: 1.0238, 9: 0.97904, 10: 0.99064,
    11: 0.98384, 12: 1.0000, 13: 1.0163, 14: 1.0300, 15: 1.0190,
    16: 0.95853, 17: 1.0174, 18: 1.0168, 19: 1.0794, 20: 1.0000,
    21: 1.0019, 22: 0.99801, 23: 0.96773, 24: 0.95610,
    25:0.85, 26:0.94
}

dataSet_to_normError = {
    1: 0.62926E-02, 2: 0.12908E-01, 3: 0.80983E-02, 4: 0.69809E-02, 5: 0.16758E-01,
    6: 0.92261E-02, 7: 0.15546E-01, 8: 0.65203E-02, 9: 0.55606E-02, 10: 0.75245E-02,
    11: 0.25318E-01, 12: 0.0, 13: 0.17632E-02, 14: 0.91993E-02, 15: 0.63181E-02,
    16: 0.25582E-01, 17: 0.42184E-01, 18: 0.68067E-01, 19: 0.35847E-01, 20: 0.0,
    21: 0.18356E-01, 22: 0.41475E-01, 23: 0.17317E-01, 24: 0.23120E-01,
    25:0.02,26:0.02
}

def round_sig_3(x):
    return '%s' % float('%.3g' % x)

def append_row(df, row):
    return pd.concat([df, pd.DataFrame([row], columns=row.index)]).reset_index(drop=True)

def linear_model(x, a, b):
    return a * x + b

# read dataframe from csv file
def prepare_df(df):

    df["Veff"] = 0.0031
    
    df["Eeff"] = df["E0"] + df["Veff"]
    df["Ep"] = df["E0"] - df["nu"]
    df["Ep_eff"] = df["Ep"] + df["Veff"]

    # calculate normalized cross section:
    df['normalization'] = df['dataSet'].map(dataSet_to_normalization)
    df['normError'] = df['dataSet'].map(dataSet_to_normError)
    df['system_err'] = 0.0
    df['normCross'] = df['cross'] * df['normalization']
    df['normCrossError'] = df['normCross'] * np.sqrt((df['error'] / df['cross'])**2 + (df['normError'] / df['normalization'])**2)
    df['normCrossError'] = np.sqrt(df['normCrossError']**2 + ((df['system_err'] * df['normCross'])**2))
    print(df.loc[df['normalization'] == 1, 'dataSet'].unique())
    
    # calculate the kinematic variables:
    df["ThetaRad"] = df["ThetaDeg"] * np.pi / 180
    df["sin2(T/2)"] = (np.sin(df["ThetaRad"] / 2))**2
    df["cos2(T/2)"] = (np.cos(df["ThetaRad"] / 2))**2
    df["tan2(T/2)"] = (np.tan(df["ThetaRad"] / 2))**2

    df["nuel"] = df["E0"] - df["E0"] / (1 + 2 * df["E0"] * df["sin2(T/2)"] / mass_nucleus)
    df["Ex"] = df["nu"] - (df["E0"] - df["E0"] / (1 + 2 * df["E0"] * df["sin2(T/2)"] / mass_nucleus))

    df["R"] = 1.1 * (df["A"])**(1/3) + 0.86 / ((df["A"])**(1/3))

    df["F2foc"] = (df["Eeff"] / df["E0"])**2
    df["Q2"] = 4 * df["E0"] * (df["Ep"]) * df["sin2(T/2)"]
    df["Q2eff"] = 4 * df["Eeff"] * df["Ep_eff"] * df["sin2(T/2)"]
    df["qv2"] = df["nu"]**2 + df["Q2eff"]
    df["qv"] = np.sqrt(df["qv2"])
    
    df["W2_O"] = mass_nucleon**2 + 2 * mass_nucleon * df["nu"] - df["Q2"]
    df["W2"] = mass_nucleon**2 + 2 * mass_nucleon * df["nu"] - df["Q2eff"]
    df["epsilon"] = 1 / (1 + 2 * (1 + (df["nu"]**2) / df["Q2eff"]) * df["tan2(T/2)"])

    df["gamma"] = alpha_fine * df["Ep_eff"] * (df["W2"] - mass_nucleon**2) / (( 4 * ((np.pi)**2) * df["Q2eff"] * mass_nucleon * df["E0"]) * (1 - df["epsilon"]))
    df["Sig_R"] = df["normCross"] / df["gamma"]
    df["D_sig_R"] = df["error"] / df["gamma"]
    df["Sig_mott"] = 4 * (alpha_fine**2) * (df["Ep"]**2) * df["cos2(T/2)"] / (df["Q2"]**2)
    df["Sig_mott_eff"] = df["Sig_mott"] * df["E0"] / df["Eeff"]

    # Calculate the Rosenbluth quantity:
    df["H"] = (df["qv2"]**2) / (4 * (alpha_fine**2) * (df["Ep_eff"]**2) * (df["cos2(T/2)"] + 2 * (df["qv2"] / df["Q2eff"]) * df["sin2(T/2)"]))
    df["Hcc"] = df["H"] / df["F2foc"]
    df["Hstar_Sig(nb)"] = df["H"] * df["normCross"]
    df["Hstar_error(nb)"] = df["H"] * df["normCrossError"]
    df["Hstar_Sig(GeV)"] = df["Hstar_Sig(nb)"] / ((0.1973269**2) * 10000000)
    df["Hstar_error(GeV)"] = df["Hstar_error(nb)"] / ((0.1973269**2) * 10000000)
    df["Hcc_Sig(nb)"] = df["Hcc"] * df["normCross"]
    df["Hcc_error(nb)"] = df["Hcc"] * df["normCrossError"]
    df["Hcc_Sig(GeV)"] = df["Hcc_Sig(nb)"] / ((0.1973269**2) * 10000000)
    df["Hcc_error(GeV)"] = df["Hcc_error(nb)"] / ((0.1973269**2) * 10000000)
    # we will fit "Hcc_Sig(GeV)" vs "epsilon" to the linear model to get the Rosenbluth slope and intercept

    # subdivide the data into bins
    df['qvbin'] = 0
    df['qvcenter'] = 0
    df['Q2bin'] = 0
    df['Q2center'] = 0
    df["qvbin"] = pd.cut(x=df["qv"], bins=qvbins, labels=qvbin_names, right=True)
    df["qvcenter"] = pd.cut(x=df["qv"], bins=qvbins, labels=qvcenters, right=True)
    df['qvcenter'] = pd.to_numeric(df['qvcenter'])
    df["Q2bin"] = pd.cut(x=df["Q2eff"], bins=Q2bins, labels=Q2bin_names, right=True)
    df["Q2center"] = pd.cut(x=df["Q2eff"], bins=Q2bins, labels=Q2centers, right=True)
    df['Q2center'] = pd.to_numeric(df['Q2center'])
    df = df.dropna()
    # now every row of data has a bin (Q2/qv) label

    # # this is for bc_qv_ex and bc_q2_ex
    # df['ExA'] = np.where(df['Ex'] >= 0.04, df['Ex'], 0.04)
    # df['nuA'] = np.where(df['Ex'] >= 0.04, df['nu'], df['nu'] + 0.04 - df['Ex'])
    # df["Q2effA"] = 4 * df["Eeff"] * (df["Eeff"] - df['nuA']) * df["sin2(T/2)"]
    # df["qv2A"] = df["nuA"]**2 + df["Q2effA"]
    # df["epsilonA"] = 1 / (1 + 2 * (1 + (df["nuA"]**2) / df["Q2effA"]) * df["tan2(T/2)"])

    Ex_qv_increments = np.array([0.0012,0.001,0.0006,0.0006,0.0006,0.0010,0.0036,0.0036,0.0036,0.0036,0.002,0.002,0.002,0.002,0.002,0.002,0.002,0.002])
    Ex_qv_to_increment = dict(zip(qvcenters, Ex_qv_increments))
    df['Ex_qv_increment'] = df['qvcenter'].map(Ex_qv_to_increment)
    df['Ex_qv_start'] = 0.0
    df['Ex_qv_bin_index'] = np.floor((df['Ex'] - df['Ex_qv_start']) / df['Ex_qv_increment']).astype(int)
    df['Ex_qvcenter'] = df['Ex_qv_start'] + (df['Ex_qv_bin_index'] + 0.5) * df['Ex_qv_increment']
    df['nu_Ex_qvcenter'] = - mass_nucleus + np.sqrt(mass_nucleus**2 + df['qvcenter']**2 + 2 * mass_nucleus * df['Ex_qvcenter'])
    df["epsilon_Ex_qvcenter"] = 1 / (1 + 2 * (1 + (df["nu_Ex_qvcenter"]**2) / (df["qvcenter"]**2 - df["nu_Ex_qvcenter"]**2)) * df["tan2(T/2)"])

    W2_qv_increments = np.array([0.01,0.002,0.002,0.014,0.018,0.01,0.008,0.032,0.06,0.018,0.04,0.12,0.12,0.12,0.12,0.24,0.24,0.24])
    W2_qv_to_increment = dict(zip(qvcenters, W2_qv_increments))
    df['W2_qv_increment'] = df['qvcenter'].map(W2_qv_to_increment)
    df['W2_qv_start'] = mass_nucleon**2 + 2 * mass_nucleon * 0.05 - (df['qvcenter']**2 - 0.05**2)
    df['W2_qv_bin_index'] = np.floor((df['W2'] - df['W2_qv_start']) / df['W2_qv_increment']).astype(int)
    df['W2_qvcenter'] = df['W2_qv_start'] + (df['W2_qv_bin_index'] + 0.5) * df['W2_qv_increment']
    df['nu_W2_qvcenter'] = np.sqrt(df['qvcenter']**2 + df['W2_qvcenter']) - mass_nucleon
    df["epsilon_W2_qvcenter"] = 1 / (1 + 2 * (1 + (df["nu_W2_qvcenter"]**2) / (df["qvcenter"]**2 - df["nu_W2_qvcenter"]**2)) * df["tan2(T/2)"])

    Ex_q2_increments = np.array([0.006,0.0025,0.005,0.005,0.003,0.005,0.005,0.005,0.005,0.005,0.005,0.005,0.005,0.005,0.005,0.005,0.005,0.005])
    Ex_q2_to_increment = dict(zip(Q2centers, Ex_q2_increments))
    df['Ex_q2_increment'] = df['Q2center'].map(Ex_q2_to_increment)
    df['Ex_q2_start'] = 0.0
    df['Ex_q2_bin_index'] = np.floor((df['Ex'] - df['Ex_q2_start']) / df['Ex_q2_increment']).astype(int)
    df['Ex_q2center'] = df['Ex_q2_start'] + (df['Ex_q2_bin_index'] + 0.5) * df['Ex_q2_increment']
    df['nu_Ex_q2center'] = df['Ex_q2center'] + df['Q2center'] / (2 * mass_nucleus)
    df["epsilon_Ex_q2center"] = 1 / (1 + 2 * (1 + (df["nu_Ex_q2center"]**2) / df["Q2center"]) * df["tan2(T/2)"])

    W2_q2_increments = np.array([0.01,0.01,0.01,0.008,0.01,0.008,0.008,0.03,0.015,0.05,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1])
    W2_q2_to_increment = dict(zip(Q2centers, W2_q2_increments))
    df['W2_q2_increment'] = df['Q2center'].map(W2_q2_to_increment)
    df['W2_q2_start'] = mass_nucleon**2 + 2 * mass_nucleon * 0.05 - df['Q2center']
    df['W2_q2_bin_index'] = np.floor((df['W2'] - df['W2_q2_start']) / df['W2_q2_increment']).astype(int)
    df['W2_q2center'] = df['W2_q2_start'] + (df['W2_q2_bin_index'] + 0.5) * df['W2_q2_increment']
    df['nu_W2_q2center'] = (df['W2_q2center'] - mass_nucleon**2 + df['Q2center']) / (2 * mass_nucleon)
    df["epsilon_W2_q2center"] = 1 / (1 + 2 * (1 + (df["nu_W2_q2center"]**2) / df["Q2center"]) * df["tan2(T/2)"])

    df = df.dropna()

    return df

In [ ]:
# read csv file
df = pd.read_csv('Data/C12.csv')
df = prepare_df(df)
df

[ 3. 25. nan 12.  4.  1.  5.  7.  8.  9. 11. 10.  2.]


C:\Users\Rhys\AppData\Local\Temp\ipykernel_25524\4051091671.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Ex_qv_increment'] = df['qvcenter'].map(Ex_qv_to_increment)
C:\Users\Rhys\AppData\Local\Temp\ipykernel_25524\4051091671.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Ex_qv_start'] = 0.0
C:\Users\Rhys\AppData\Local\Temp\ipykernel_25524\4051091671.py:139: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexe

,Z,A,E0,ThetaDeg,nu,cross,error,dataSet,Veff,Eeff,...,Ex_q2_bin_index,Ex_q2center,nu_Ex_q2center,epsilon_Ex_q2center,W2_q2_increment,W2_q2_start,W2_q2_bin_index,W2_q2center,nu_W2_q2center,epsilon_W2_q2center
1,6,12,1.108,37.5,0.016620,0.141794,0.141794,3.0,0.0031,1.1111,...,-2,-0.00750,0.014866,0.812638,0.10,0.474184,-1,0.424184,0.023355,0.812539
2,6,12,1.108,37.5,0.027700,1.985120,0.530546,3.0,0.0031,1.1111,...,1,0.00750,0.029866,0.812434,0.10,0.474184,-1,0.424184,0.023355,0.812539
3,6,12,1.108,37.5,0.038780,11.343500,1.268250,3.0,0.0031,1.1111,...,3,0.01750,0.039866,0.812222,0.10,0.474184,-1,0.424184,0.023355,0.812539
4,6,12,1.108,37.5,0.049860,27.508100,1.974970,3.0,0.0031,1.1111,...,5,0.02750,0.049866,0.811949,0.10,0.474184,0,0.524184,0.076645,0.810921
5,6,12,1.108,37.5,0.060940,50.762400,2.682880,3.0,0.0031,1.1111,...,7,0.03750,0.059866,0.811616,0.10,0.474184,0,0.524184,0.076645,0.810921
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26482,6,12,0.961,37.5,0.879315,3163.910000,20.952600,3.0,0.0031,0.9641,...,172,0.86250,0.863663,0.127517,0.01,0.948184,154,2.493184,0.873321,0.125144
26483,6,12,0.961,37.5,0.888925,3199.710000,21.070800,3.0,0.0031,0.9641,...,174,0.87250,0.873663,0.125061,0.01,0.948184,157,2.523184,0.889308,0.121351
26484,6,12,0.961,37.5,0.898535,3257.160000,21.259100,3.0,0.0031,0.9641,...,176,0.88250,0.883663,0.122672,0.01,0.948184,159,2.543184,0.899966,0.118912
26485,6,12,0.961,37.5,0.908145,2459.310000,18.472800,3.0,0.0031,0.9641,...,356,0.89125,0.892145,0.096137,0.01,0.954184,160,2.559184,0.905295,0.093684


In [4]:
response_columns = ['i','RTTOT','RLTOT','RTQE','RLQE','RTNS','RLNS']

# calculate bc_qv_ex
df[['qvcenter','Ex_qvcenter']].to_csv('input.txt',index=True,header=False,sep=' ')
with open('output.txt', 'w') as output_file:
    subprocess.run(['./response_qv_ex.exe', 'input.txt'], stdout=output_file) 
subprocess.run(['sleep', '0.5'])
df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
df['RL_qvc_ex'] = df.index.map(df_response.set_index('i')['RLTOT'])
df['RT_qvc_ex'] = df.index.map(df_response.set_index('i')['RTTOT'])

df[['qv','Ex']].to_csv('input.txt',index=True,header=False,sep=' ')
with open('output.txt', 'w') as output_file:
    subprocess.run(['./response_qv_ex.exe', 'input.txt'], stdout=output_file) 
subprocess.run(['sleep', '0.5'])
df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
df['RL_qvd_ex'] = df.index.map(df_response.set_index('i')['RLTOT'])
df['RT_qvd_ex'] = df.index.map(df_response.set_index('i')['RTTOT'])
print('RL RT qv_ex done.')

# calculate bc_qv_w2
df[['qvcenter','W2_qvcenter']].to_csv('input.txt',index=True,header=False,sep=' ')
with open('output.txt', 'w') as output_file:
    subprocess.run(['./response_qv_w2.exe', 'input.txt'], stdout=output_file) 
subprocess.run(['sleep', '0.5'])
df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
df['RL_qvc_w2'] = df.index.map(df_response.set_index('i')['RLTOT'])
df['RT_qvc_w2'] = df.index.map(df_response.set_index('i')['RTTOT'])

df[['qv','W2']].to_csv('input.txt',index=True,header=False,sep=' ')
with open('output.txt', 'w') as output_file:
    subprocess.run(['./response_qv_w2.exe', 'input.txt'], stdout=output_file) 
subprocess.run(['sleep', '0.5'])
df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
df['RL_qvd_w2'] = df.index.map(df_response.set_index('i')['RLTOT'])
df['RT_qvd_w2'] = df.index.map(df_response.set_index('i')['RTTOT'])
print('RL RT qv_w2 done.')

# calculate bc_q2_ex
df[['Q2center','Ex_q2center']].to_csv('input.txt',index=True,header=False,sep=' ')
with open('output.txt', 'w') as output_file:
    subprocess.run(['./response_q2_ex.exe', 'input.txt'], stdout=output_file) 
subprocess.run(['sleep', '0.5'])
df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
df['RL_q2c_ex'] = df.index.map(df_response.set_index('i')['RLTOT'])
df['RT_q2c_ex'] = df.index.map(df_response.set_index('i')['RTTOT'])

df[['Q2eff','Ex']].to_csv('input.txt',index=True,header=False,sep=' ')
with open('output.txt', 'w') as output_file:
    subprocess.run(['./response_q2_ex.exe', 'input.txt'], stdout=output_file) 
subprocess.run(['sleep', '0.5'])
df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
df['RL_q2d_ex'] = df.index.map(df_response.set_index('i')['RLTOT'])
df['RT_q2d_ex'] = df.index.map(df_response.set_index('i')['RTTOT'])
print('RL RT q2_ex done.')

# calculate bc_q2_w2
df[['Q2center','W2_q2center']].to_csv('input.txt',index=True,header=False,sep=' ')
with open('output.txt', 'w') as output_file:
    subprocess.run(['./response_q2_w2.exe', 'input.txt'], stdout=output_file) 
subprocess.run(['sleep', '0.5'])
df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
df['RL_q2c_w2'] = df.index.map(df_response.set_index('i')['RLTOT'])
df['RT_q2c_w2'] = df.index.map(df_response.set_index('i')['RTTOT'])

df[['Q2eff','W2']].to_csv('input.txt',index=True,header=False,sep=' ')
with open('output.txt', 'w') as output_file:
    subprocess.run(['./response_q2_w2.exe', 'input.txt'], stdout=output_file) 
subprocess.run(['sleep', '0.5'])
df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=response_columns)
df['RL_q2d_w2'] = df.index.map(df_response.set_index('i')['RLTOT'])
df['RT_q2d_w2'] = df.index.map(df_response.set_index('i')['RTTOT'])
print('RL RT q2_w2 done.')


<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:16: SyntaxWarning: invalid escape sequence '\s'
<>:26: SyntaxWarning: invalid escape sequence '\s'
<>:34: SyntaxWarning: invalid escape sequence '\s'
<>:44: SyntaxWarning: invalid escape sequence '\s'
<>:52: SyntaxWarning: invalid escape sequence '\s'
<>:62: SyntaxWarning: invalid escape sequence '\s'
<>:70: SyntaxWarning: invalid escape sequence '\s'
<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:16: SyntaxWarning: invalid escape sequence '\s'
<>:26: SyntaxWarning: invalid escape sequence '\s'
<>:34: SyntaxWarning: invalid escape sequence '\s'
<>:44: SyntaxWarning: invalid escape sequence '\s'
<>:52: SyntaxWarning: invalid escape sequence '\s'
<>:62: SyntaxWarning: invalid escape sequence '\s'
<>:70: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Rhys\AppData\Local\Temp\ipykernel_25524\3851954695.py:8: SyntaxWarning: invalid escape sequence '\s'
  df_response = pd.read_csv('output.txt', sep='\s+', header=None, names=resp

RL RT qv_ex done.
RL RT qv_w2 done.
RL RT q2_ex done.
RL RT q2_w2 done.


In [5]:
# determine bin-centering corrections
df["bc_qv_ex"] = 1.0
df["bc_qv_w2"] = 1.0
df["bc_q2_ex"] = 1.0
df["bc_q2_w2"] = 1.0
for qvcenter in qvcenters:
    picked = df.loc[df["qvcenter"] == qvcenter]
    for index, row in picked.iterrows():
        if row['qvcenter'] != float('NaN'):
            try:
                df.loc[index, 'bc_qv_ex'] = (row['epsilon_Ex_qvcenter'] * row['RL_qvc_ex'] + 0.5 * ((qvcenter**2) / (qvcenter**2 - row['nu_Ex_qvcenter']**2)) * row['RT_qvc_ex']) / (row['epsilon'] * row['RL_qvd_ex'] + 0.5 * (row['qv2'] / row['Q2eff']) * row['RT_qvd_ex'])
            except ZeroDivisionError:
                print(f'Skipping index {index} for bc_qv_ex due to zero denominator.')
            try:
                df.loc[index, 'bc_qv_w2'] = (row['epsilon_W2_qvcenter'] * row['RL_qvc_w2'] + 0.5 * ((qvcenter**2) / (qvcenter**2 - row['nu_W2_qvcenter']**2)) * row['RT_qvc_w2']) / (row['epsilon'] * row['RL_qvd_w2'] + 0.5 * (row['qv2'] / row['Q2eff']) * row['RT_qvd_w2'])
            except ZeroDivisionError:
                print(f'Skipping index {index} for bc_qv_w2 due to zero denominator.')

for Q2center in Q2centers:
    picked = df.loc[df["Q2center"] == Q2center]
    for index, row in picked.iterrows():
        if row['Q2center'] != float('NaN'):
            try:
                df.loc[index, 'bc_q2_ex'] = (row['epsilon_Ex_q2center'] * row['RL_q2c_ex'] + 0.5 * ((Q2center + row["nu_Ex_q2center"]**2) / (Q2center)) * row['RT_q2c_ex']) / (row['epsilon'] * row['RL_q2d_ex'] + 0.5 * (row['qv2'] / row['Q2eff']) * row['RT_q2d_ex'])
            except ZeroDivisionError:
                print(f'Skipping index {index} for bc_q2_ex due to zero denominator.')
            try:
                df.loc[index, 'bc_q2_w2'] = (row['epsilon_W2_q2center'] * row['RL_q2c_w2'] + 0.5 * ((Q2center + row["nu_W2_q2center"]**2) / (Q2center)) * row['RT_q2c_w2']) / (row['epsilon'] * row['RL_q2d_w2'] + 0.5 * (row['qv2'] / row['Q2eff']) * row['RT_q2d_w2'])
            except ZeroDivisionError:
                print(f'Skipping index {index} for bc_q2_w2 due to zero denominator.')

Skipping index 1 for bc_qv_ex due to zero denominator.
Skipping index 1 for bc_qv_w2 due to zero denominator.
Skipping index 2 for bc_qv_ex due to zero denominator.
Skipping index 2 for bc_qv_w2 due to zero denominator.
Skipping index 99 for bc_qv_ex due to zero denominator.
Skipping index 99 for bc_qv_w2 due to zero denominator.
Skipping index 100 for bc_qv_ex due to zero denominator.
Skipping index 100 for bc_qv_w2 due to zero denominator.
Skipping index 1429 for bc_qv_ex due to zero denominator.
Skipping index 1429 for bc_qv_w2 due to zero denominator.
Skipping index 6031 for bc_qv_ex due to zero denominator.
Skipping index 6031 for bc_qv_w2 due to zero denominator.
Skipping index 9109 for bc_qv_ex due to zero denominator.
Skipping index 9109 for bc_qv_w2 due to zero denominator.
Skipping index 9489 for bc_qv_ex due to zero denominator.
Skipping index 9489 for bc_qv_w2 due to zero denominator.
Skipping index 12607 for bc_qv_ex due to zero denominator.
Skipping index 12607 for bc_qv_

In [ ]:
# save dataframe to csv
df.to_csv('Data/df_C12.csv',index=False)